# Reading from bronze

In [0]:
import pyspark.sql.functions as F 
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim 
df = spark.table("databricksprash.bronze_layer_data.crm_cust_info")

###Trimming

In [0]:
for field in df.schema.fields :
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

###Normalisation

In [0]:
# Normalisation 
df = ( 
      df
      .withColumn(
          "cst_marital_status", 
          F.when(F.upper(F.col("cst_marital_status")) =="S", "Single")
          .when(F.upper(F.col("cst_marital_status")) =="M", "Married")
          .otherwise(F.col("cst_marital_status")) 
      )
      .withColumn(
          "cst_gndr", 
          F.when(F.upper(F.col("cst_gndr")) =="M", "Male")
          .when(F.upper(F.col("cst_gndr")) =="F", "Female")
          .otherwise(F.col("cst_gndr"))
      )
)

# df.display()    

### Remove Records with Missing Customer ID 

In [0]:
df = df.filter(col("cst_id").isNotNull())

###Renaming Columns

In [0]:
#Renaming Map 
RENAME_MAP = {
 "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}

#Renaming columns ussing map
for old_name, new_name in RENAME_MAP.items():
  df = df.withColumnRenamed(old_name, new_name)
df.limit(10).display()

In [0]:
# space cleaning
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))


###Sanity Check for dataframe



In [0]:
df.limit(10).display()

Write into silver table

In [0]:

df.write.mode("overwrite").format("delta").saveAsTable("silver_layer_data.customer_info")

In [0]:
%sql
select * from silver_layer_data.customer_info
limit 10;